In [ ]:
import os
from typing import Any
import json
import time
import uuid
from pathlib import Path


import mlflow
from datasets import concatenate_datasets, load_dataset
from mlflow.entities import Feedback
from mlflow.genai import evaluate, load_prompt, optimize_prompts, register_prompt


import langextract as lx
from rich.pretty import pprint

from pathlib import Path
from typing import Iterable

import requests

import re 


## Langextract Configuration

In [ ]:
# 1. Define the prompt and extraction rules
prompt = """
    Sos un asistente especializado en el análisis de documentos judiciales.
    Tu tarea es identificar y extraer menciones de información sensible para su posterior anonimización.
    Debés detectar fragmentos textuales que correspondan a cualquiera de las siguientes entidades:

    - "BANCO": Nombre de una entidad bancaria, pública o privada.
    - "CBU": Código Bancario Uniforme (22 dígitos) de una cuenta.
    - "CORREO_ELECTRONICO": Dirección de correo electrónico.
    - "CUIJ": Código Único de Identificación Jurídica de causas judiciales.
    - "CUIT_CUIL": Número de CUIT o CUIL de una persona física o jurídica.
    - "DIRECCION": Dirección postal específica (calle, número, etc.).
    - "DNI": Número de Documento Nacional de Identidad u otro documento identificatorio.
    - "EDAD": Edad explícita de una persona.
    - "ESTUDIOS": Nivel o institución educativa que permita identificar a la persona (ej. "primario incompleto", "secundario completo", "Licenciado en…").
    - "FECHA": Fecha completa o parcial (día, mes y/o año).
    - "LINK": Enlace o URL a una página web.
    - "LOC": Localización geográfica específica (ciudad, barrio, comisaría, etc.).
    - "MARCA_AUTOMOVIL": Marca de un vehículo (ej. Toyota, Ford).
    - "NACIONALIDAD": Nacionalidad de una persona (ej. "argentino", "brasileña").
    - "NUM_ACTUACION": Número identificatorio de una actuación administrativa o contravencional.
    - "NUM_CAJA_AHORRO": Número completo de una caja de ahorro o cuenta bancaria.
    - "NUM_EXPEDIENTE": Número de expediente judicial o administrativo.
    - "NUM_MATRICULA": Número de matrícula profesional o académica.
    - "PATENTE_DOMINIO": Patente o dominio de un vehículo.
    - "PER": Nombre y apellido(s) de una persona física. Los nombres inicializados y los apodos también cuentan como información sensible a anonimizar.
    - "TELEFONO": Número telefónico (fijo o celular).
"""

In [ ]:
# 2. Provide a high-quality example to guide the model
examples = [
    lx.data.ExampleData(
        text="El 5 de mayo de 2023 el señor Fiscal indicó que realizó distintas medidas de prueba y que del resultado surge que tanto la investigada como el menor Juan Pérez se domicilian en la calle Sarmiento 1234, de la localidad de Moreno, por lo que solicitó que se declare la incompetencia en razón del territorio y se envíe el caso al Juzgado de Garantías que corresponda del Departamento Judicial de Moreno, con jurisdicción en el partido de Moreno.",
        extractions=[
            lx.data.Extraction(
                extraction_class="FECHA", extraction_text="5 de mayo de 2023"
            ),
            lx.data.Extraction(extraction_class="PER", extraction_text="Juan Pérez"),
            lx.data.Extraction(
                extraction_class="DIRECCION", extraction_text="Sarmiento 1234"
            ),
            lx.data.Extraction(extraction_class="LOC", extraction_text="Moreno"),
        ],
    ),
    lx.data.ExampleData(
        text="JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVENCIONAL Y DE FALTAS N°10 SECRETARIA N°19\nCarlos Gómez sobre 84 - HOMICIDIO CULPOSO Y OTROS\nNúmero: 52345/2022\nCUIJ: 12-34567890-1\nActuación Nro: 2022-009876",
        extractions=[
            lx.data.Extraction(extraction_class="PER", extraction_text="Carlos Gómez"),
            lx.data.Extraction(
                extraction_class="NUM_EXPEDIENTE", extraction_text="52345/2022"
            ),
            lx.data.Extraction(
                extraction_class="CUIJ", extraction_text="12-34567890-1"
            ),
            lx.data.Extraction(
                extraction_class="NUM_ACTUACION", extraction_text="2022-009876"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Acusado: Miguel Torres, DNI 30123456, nacido el 14/02/1990, de 34 años de edad, de nacionalidad paraguaya, género varón cis, con estudios secundarios completos, hizo hasta 3er año porque fue padre joven, con último domicilio en Av. Corrientes 3456, de esta ciudad, donde vive con su hermana y su cuñado Jorge Pérez. Tiene dos hijos a su cargo, de 5 y 8 años. Su hijo de 8 vive con él, su hija de 5 vive con su madre, Laura Fernández.",
        extractions=[
            lx.data.Extraction(extraction_class="PER", extraction_text="Miguel Torres"),
            lx.data.Extraction(extraction_class="DNI", extraction_text="30123456"),
            lx.data.Extraction(extraction_class="FECHA", extraction_text="14/02/1990"),
            lx.data.Extraction(extraction_class="EDAD", extraction_text="34"),
            lx.data.Extraction(
                extraction_class="NACIONALIDAD", extraction_text="paraguaya"
            ),
            lx.data.Extraction(
                extraction_class="ESTUDIOS",
                extraction_text="estudios secundarios completos",
            ),
            lx.data.Extraction(
                extraction_class="DIRECCION",
                extraction_text="Av. Corrientes 3456",
            ),
            lx.data.Extraction(extraction_class="PER", extraction_text="Jorge Pérez"),
            lx.data.Extraction(extraction_class="EDAD", extraction_text="5"),
            lx.data.Extraction(extraction_class="EDAD", extraction_text="8"),
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Laura Fernández"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="El testigo Juan López dejó asentado su número de contacto: 11-2345-6789. Indicó que la médica Dra. Ana García, MN 12345, asistió al lugar donde se hallaba un vehículo Volkswagen, patente AB123CD.",
        extractions=[
            lx.data.Extraction(extraction_class="PER", extraction_text="Juan López"),
            lx.data.Extraction(
                extraction_class="TELEFONO", extraction_text="11-2345-6789"
            ),
            lx.data.Extraction(extraction_class="PER", extraction_text="Ana García"),
            lx.data.Extraction(
                extraction_class="NUM_MATRICULA", extraction_text="12345"
            ),
            lx.data.Extraction(
                extraction_class="MARCA_AUTOMOVIL",
                extraction_text="Volkswagen",
            ),
            lx.data.Extraction(
                extraction_class="PATENTE_DOMINIO", extraction_text="AB123CD"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Se identificó una transferencia bancaria con los siguientes datos: CUIT 20-12345678-3, CBU 2850590940090412345671, Caja de Ahorro N° 12345678, Banco Nación.",
        extractions=[
            lx.data.Extraction(
                extraction_class="CUIT_CUIL", extraction_text="20-12345678-3"
            ),
            lx.data.Extraction(
                extraction_class="CBU",
                extraction_text="2850590940090412345671",
            ),
            lx.data.Extraction(
                extraction_class="NUM_CAJA_AHORRO", extraction_text="12345678"
            ),
            lx.data.Extraction(
                extraction_class="BANCO", extraction_text="Banco Nación"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Para mayor información, comunicarse a fiscalia.central@justicia.gob.ar o visitar el sitio https://justicia.gob.ar/actuaciones.",
        extractions=[
            lx.data.Extraction(
                extraction_class="CORREO_ELECTRONICO",
                extraction_text="fiscalia.central@justicia.gob.ar",
            ),
            lx.data.Extraction(
                extraction_class="LINK",
                extraction_text="https://justicia.gob.ar/actuaciones",
            ),
        ],
    ),
    lx.data.ExampleData(
        text="En la Ciudad Autónoma de Buenos Aires, el día 5 de mayo de 2023, el Sr. Fiscal hace saber que Juan Pérez se domicilia en la calle Sarmiento 1234, localidad de Moreno.",
        extractions=[
            lx.data.Extraction(
                extraction_class="FECHA", extraction_text="5 de mayo de 2023"
            ),
            lx.data.Extraction(extraction_class="PER", extraction_text="Juan Pérez"),
            lx.data.Extraction(
                extraction_class="DIRECCION", extraction_text="Sarmiento 1234"
            ),
            lx.data.Extraction(extraction_class="LOC", extraction_text="Moreno"),
        ],
    ),
    lx.data.ExampleData(
        text="Por otra parte, la División Investigaciones Judiciales de la Policía Federal Argentina informó que no se dio intervención a ninguna otra Fiscalía u otro Juzgado por la sustracción del vehículo Volkswagen Voyage, dominio KXY-876",
        extractions=[
            lx.data.Extraction(
                extraction_class="PATENTE_DOMINIO", extraction_text="KXY-876"
            ),
            lx.data.Extraction(
                extraction_class="MARCA_AUTOMOVIL",
                extraction_text="Volkswagen Voyage",
            ),
        ],
    ),
    lx.data.ExampleData(
        text="A su vez, requirió informes al Banco BBVA Francés, respecto de las cuentas bancarias de la denunciante, Carla Alejandra Garcia, D.N.I. 36.998.621 identificadas como Caja de ahorro en pesos argentinos número 117-59824/6 con CBU 0180132640000004685591 y Caja de ahorro en dólares número 119-619018/2 con CBU 0170115544000062081822.",
        extractions=[
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Carla Alejandra Garcia"
            ),
            lx.data.Extraction(extraction_class="DNI", extraction_text="36.998.621"),
            lx.data.Extraction(
                extraction_class="NUM_CAJA_AHORRO", extraction_text="117-59824/6"
            ),
            lx.data.Extraction(
                extraction_class="CBU", extraction_text="0180132640000004685591"
            ),
            lx.data.Extraction(
                extraction_class="NUM_CAJA_AHORRO", extraction_text="119-619018/2"
            ),
            lx.data.Extraction(
                extraction_class="CBU", extraction_text="0170115544000062081822"
            ),
        ],
    ),
]

In [ ]:
# Sample text to extract from
text = "La Fiscalía determinó que el objeto de este caso es investigar el hecho que tuvo lugar el día 12 de marzo de 2023 a las 8:50 horas aproximadamente, ocasión en que Carlos Gómez y María Rodriguez estafaron a Juan Pérez por un monto total de pesos treinta y tres mil novecientos sesenta ($33.960)."

# Run the extraction
result = lx.extract(
    text_or_documents=text,
    prompt_description=prompt,
    examples=examples,
    model_id="phi4:14b",
    model_url="http://localhost:11434",
    max_char_buffer=16_384,
    fence_output=False,
    use_schema_constraints=False,
)

pprint(result)

## MLflow Configuration

In [ ]:
# ========================================
# Configuración de MLflow y Ollama
# ========================================

# MLflow: servidor de tracking y modelo registry
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:8080")
EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT", "NER-langextract-comparison")

# Ollama: proveedor de LLM local (OpenAI-compatible API)
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma3:270m")
OLLAMA_REFLECTION_MODEL = os.getenv("OLLAMA_REFLECTION_MODEL", "llama3.1:8b")

# ========================================
# Inicialización de MLflow
# ========================================

# Define la URI de seguimiento y el experimento en MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

# Habilita trazas automáticas para llamadas OpenAI
# Como Ollama implementa la API de OpenAI, esto también captura llamadas a Ollama
mlflow.openai.autolog()

## NER Configuration document-extract endpoint output

In [ ]:
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
ENDPOINT_DOCUMENT_EXTRACT = f"{API_BASE_URL}/misc/document-extract"

DATA_ROOT = Path(
    os.getenv(
        "DOCUMENT_DATA_ROOT", "../../../resources/data/restricted/disambiguation-eval/files"
    )
)

DOC_EXTENSIONS = {".pdf", ".docx"}
JSON_EXTENSION = {".json"}

REQUEST_TIMEOUT_S = float(os.getenv("DOCUMENT_REQUEST_TIMEOUT", "30"))

print(f"Target endpoint: {ENDPOINT_DOCUMENT_EXTRACT}")
print(f"Data root: {DATA_ROOT.resolve()}")

print(f"API: {API_BASE_URL}")
print(f"Data root: {DATA_ROOT}")

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Directory '{DATA_ROOT}' not found. Update DATA_ROOT before continuing."
    )


def discover_documents(root: Path, extensions: Iterable[str]) -> list[Path]:
    extensions = {ext.lower() for ext in extensions}
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )


documents = discover_documents(DATA_ROOT, DOC_EXTENSIONS)

## Functions for API endpoints usage

In [ ]:
from aymurai.experiments.entity_disambiguation.runner import (
    call_extraction_api as extract_document,
)

def _extract_document(document_path: Path) -> Any:

    clean_base_name = document_path.stem.replace("_", "-").replace(" ", "-")
    target_filename = re.sub(r"-{2,}", "-", clean_base_name).strip("-")

    print(f"----- Processing document: {target_filename} -----")

    session = requests.Session()
    api_response = extract_document(
        session,
        file_path = Path(document_path),
        endpoint=f"{API_BASE_URL}/misc/document-extract",
        timeout_s=300,
    )

    document_paragraphs = api_response.get("detail", {}).get("document")

    return document_paragraphs

def _ner_predictions(paragraphs: list[dict]) -> dict:

    endpoint = f"{API_BASE_URL}/anonymizer/predict"
    
    print("----- Processing NER predictions -----")
    ner_predictions = {}

    with mlflow.start_span(name="ner_batch_inference") as parent_span:
        for i, paragraph in enumerate(paragraphs):
            with mlflow.start_span(name=f"ner_p{i}") as span:
                response = requests.post(endpoint, json={"text": paragraph}, timeout=60)
                response_data = response.json()
                span.set_inputs({"text": paragraph})
                span.set_outputs(response_data)

                if response_data.get("labels"):
                    for label in response_data.get("labels", []):
                        attrs = label.get("attrs", {})
                        text = label.get("aymurai_alt_text", label.get("text"))
                        aymurai_label = attrs.get("aymurai_label")
                        
                        ner_predictions[text] = aymurai_label

    print(f"----- Got {len(ner_predictions)} NER predictions. -----")
    
    return ner_predictions

def _lx_predictions(paragraphs: list[dict]) -> dict:

    print("----- Processing document with langextract -----")
    lx_predictions = {}

    with mlflow.start_span(name="langextract_batch_inference") as parent_span:
        for i, paragraph in enumerate(paragraphs):
            with mlflow.start_span(name=f"phi4_p{i}") as span:
                span.set_inputs({"text": paragraph, "model": "phi4:14b"})
                
                result = lx.extract(
                    text_or_documents=paragraph,
                    prompt_description=prompt,
                    examples=examples,
                    model_id="phi4:14b",
                    model_url="http://localhost:11434",
                    max_char_buffer=16_384,
                    fence_output=False,
                    use_schema_constraints=False,
                )
                
                span.set_outputs({"extractions": [str(e) for e in result.extractions]})

                for item in result.extractions:
                    text = item.extraction_text
                    aymurai_label = item.extraction_class
                    
                    lx_predictions[text] = aymurai_label
    
    print(f"----- Got {len(lx_predictions)} langextract predictions. -----")

    return lx_predictions

@mlflow.trace(name="document_processing_trace")
def predict_document(document_path: Path) -> list[dict]:

    document_paragraphs = _extract_document(document_path)
    
    ner_preds = _ner_predictions(document_paragraphs)
    lx_preds = _lx_predictions(document_paragraphs)

    matches = []
    print("\n----- Comparing NER and LangExtract predictions -----")
    for ner_text, ner_label in ner_preds.items():
            for lx_text, lx_label in lx_preds.items():
                if lx_text.lower() in ner_text.lower() and ner_label == lx_label:
                    matches.append({
                        "matched_text": lx_text,
                        "full_ner_text": ner_text,
                        "label": ner_label
                    })

    print(f"----- Found {len(matches)} matching predictions. -----")
    
    return {
        "filename": document_path.name,
        "ner": ner_preds,
        "lx": lx_preds,
        "matches": matches
    }

In [ ]:
preds = predict_document(documents[1])

In [ ]:
preds

## MLflow Experimentation

In [ ]:
# Prompt name for registration in MLflow
PROMPT_NAME = "langextract-judicial-entities-v1"

register_prompt(
    name=PROMPT_NAME,
    template=prompt,
    commit_message="Prompt base para clasificación de entidades judiciales",
    tags={"task": "judicial-entities", "dataset": "judicial", "version": "baseline"},
)

In [ ]:
def run_benchmark(document_paths: list[Path]):
    
    # Start Parent Run
    with mlflow.start_run(run_name=f"Batch_Process_{uuid.uuid4().hex[:6]}"):
        mlflow.log_params({
            "model_id": "phi4:14b",
            "context_window": 2096,
            "ner_endpoint": f"{API_BASE_URL}/anonymizer/predict"
        })
        
        # Log the prompt used for LangExtract as an artifact
        mlflow.log_text(prompt, "prompts/extraction_prompt.txt")

        for doc_path in document_paths:
            with mlflow.start_run(run_name=doc_path.name, nested=True):
                print(f"\n>>> Processing: {doc_path.name}")
                start_time = time.time()
                
                try:
                    results = predict_document(doc_path)
                    duration = time.time() - start_time
                    
                    # --- Log Metrics ---
                    mlflow.log_metric("duration_sec", duration)
                    mlflow.log_metric("ner_entities_found", len(results["ner"]))
                    mlflow.log_metric("lx_entities_found", len(results["lx"]))
                    mlflow.log_metric("matches_count", len(results["matches"]))
                    
                    # Simple accuracy metric: matches vs total LangExtract detections
                    match_rate = len(results["matches"]) / len(results["ner"]) if results["ner"] else 0
                    mlflow.log_metric("match_rate", match_rate)

                    # --- Log Artifacts ---
                    # Save detailed results to a JSON file
                    output_dir = Path("comparision-results")
                    output_dir.mkdir(parents=True, exist_ok=True)
                    json_file = output_dir / f"results_{doc_path.stem}.json"

                    with open(json_file, "w") as f:
                        json.dump(results, f, indent=4)
                    
                    mlflow.log_artifact(json_file)
                    
                    print(f"Done. Match Rate: {match_rate:.2%}")

                except Exception as e:
                    print(f"Error processing {doc_path.name}: {e}")
                    mlflow.log_param("error_message", str(e))

In [ ]:
documents_to_process = documents[1:2]

if not os.path.exists(DATA_ROOT):
    print(f"Error: The directory '{DATA_ROOT}' was not found.")
else:
    print(f"Starting Experimentation on folder: {DATA_ROOT}")
    try:
        run_benchmark(documents_to_process)
        print("\nExperimentation complete! Open the MLflow UI to analyze data.")
    except Exception as e:
        print(f"Critical Error during run: {e}")